```
========================================================================
                      [ NOTICIA / TEXTO COMPLETO ]
========================================================================
                                  |
                                  v
                            [ MEC GEN ]
                                  |
      +---------------------------+---------------------------+
      | (Extrae párrafos individuales: Párrafo 1, Párrafo 2, Párrafo 3...)
      v
========================================================================
                   [ MODELO CLASIFICADOR PRINCIPAL ] 
                      (Detección de Falsedad)
========================================================================
                                  |
      +---------------------------+---------------------------+
      |                                                       |
      v                                                       v
[ ANÁLISIS GENERAL ]                                 [ ANÁLISIS POR PÁRRAFO ]
- Noticia en General: FALSA                          - P1: Falso (Score: 0.8)
                                                     - P2: Falso (Score: 0.7)
                                                     - P3: Verdadero (Score: 0.65)
                                                              |
                                                              v
                                             (Filtro: Pasan los párrafos 
                                              con el SCORE MÁS ALTO)
                                                              |
                                  +---------------------------+
                                  |
                                  v
                [ PÁRRAFOS FILTRADOS + NOTICIA COMPLETA ]
                                  |
      +---------------------------+---------------------------+
      |                                                       |
      v                                                       v
=============================                   =============================
[   MODELO CLASIFICADOR     ]                   [   MODELO CLASIFICADOR     ]
[     SENSACIONALISMO       ]                   [       REDUNDANCIA         ]
=============================                   =============================
      |                                                       |
      +-> Texto General:                                      +-> Texto General:
      |   [ Sensacionalista ]                                 |   [ Redundante ]
      |                                                       |
      +-> Párrafo 1 (P1):                                     +-> Párrafo 1 (P1):
      |   [ NO Sensacional ] (Score: 0.13)                    |   [ NO Redundante ] (Score: 0.3)
      |                                                       |
      +-> Párrafo 3 (P3):                                     +-> Párrafo 3 (P3):
          [ Sensacionalista ] (Score: 0.9)                        [ Redundante ] (Score: 0.8)
```


# 📦 1. Instalación de Dependencias y Descarga de Recursos NLP
En esta celda se instalan y descargan las librerías necesarias para ejecutar el pipeline de modelos en **Google Colab** o en un entorno local.


In [ ]:
# Instalación de librerías para PyTorch, Transformers, SentenceTransformers, SpaCy y NLTK
!pip install -q torch transformers sentence-transformers spacy nltk pandas numpy scikit-learn polars scipy

import nltk
import spacy

print("📥 Descargando recursos de NLTK y modelo de spaCy en español...")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

try:
    nlp = spacy.load("es_core_news_sm")
except Exception:
    import os
    os.system("python -m spacy download es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")

print("✅ Entorno preparado correctamente con todas las librerías necesarias.")


# ⚙️ 2. Importación de Módulos e Inicialización de Dispositivo (GPU/CPU)
Importación de módulos requeridos y verificación del dispositivo de cómputo (CUDA GPU o CPU).


In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
from sentence_transformers import SentenceTransformer, util

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Dispositivo seleccionado para el pipeline: {device}")
if device.type == 'cuda':
    print(f"🚀 GPU Activa: {torch.cuda.get_device_name(0)}")


# 🤖 3. Carga e Inicialización de los Modelos del Pipeline
En esta sección cargamos los 3 modelos principales que integran el pipeline de investigación:
1. **SaBERT**: `VerificadoProfesional/SaBERT-Spanish-Fake-News` (Modelo clasificador principal de falsedad/veracidad).
2. **Bert Spanish Sensationalism**: `JJNeila/bert-spanish-sensationalism-oss` (Modelo detector de sensacionalismo).
3. **SBERT Clásico sobre BETO**: `hiiamsid/sentence_similarity_spanish_es` (Modelo nativo en español d=768 para análisis de redundancia).


In [ ]:
print("⏳ Cargando Modelo 1: SaBERT (Detección de Falsedad)...")
sabert_tokenizer = BertTokenizer.from_pretrained("VerificadoProfesional/SaBERT-Spanish-Fake-News")
sabert_model = BertForSequenceClassification.from_pretrained("VerificadoProfesional/SaBERT-Spanish-Fake-News").to(device)
sabert_model.eval()
print("✅ SaBERT cargado exitosamente.")

print("\n⏳ Cargando Modelo 2: Clasificador de Sensacionalismo...")
sensacionalismo_classifier = pipeline(
    "text-classification",
    model="JJNeila/bert-spanish-sensationalism-oss",
    tokenizer="JJNeila/bert-spanish-sensationalism-oss",
    device=0 if torch.cuda.is_available() else -1,
    top_k=None
)
print("✅ Clasificador de Sensacionalismo cargado exitosamente.")

print("\n⏳ Cargando Modelo 3: SentenceTransformers (BETO SBERT para Redundancia)...")
sbert_model = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es', device=str(device))
print("✅ SBERT cargado exitosamente.")


# 🛠️ 4. Definición de Funciones de Procesamiento (MEC GEN, SaBERT, Sensacionalismo y Redundancia)
Definición de submódulos:
- `mec_gen_extraer_parrafos`: Descompone la noticia en párrafos u oraciones individuales.
- `evaluar_sabert_falsedad`: Evalúa falsedad/veracidad y probabilidades con SaBERT.
- `evaluar_sensacionalismo`: Mide el grado de sensacionalismo del texto.
- `evaluar_redundancia`: Determina similitud semántica y redundancia informativa.


In [ ]:
def mec_gen_extraer_parrafos(texto):
    """
    Extracción e identificación de párrafos u oraciones individuales (MEC GEN).
    """
    if not isinstance(texto, str) or not texto.strip():
        return []
    
    lineas = [p.strip() for p in texto.split("\n") if len(p.strip()) > 10]
    if len(lineas) >= 2:
        return lineas
    
    doc = nlp(texto)
    oraciones = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 15]
    if len(oraciones) >= 2:
        return oraciones
    
    fragmentos = [frag.strip() for frag in texto.replace('\n', ' ').split('.') if len(frag.strip()) > 15]
    return fragmentos if fragmentos else [texto.strip()]

def evaluar_sabert_falsedad(texto):
    """
    Clasifica si un texto o párrafo es FALSO o VERDADERO usando SaBERT.
    """
    inputs = sabert_tokenizer(str(texto), return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = sabert_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).squeeze().tolist()
    if isinstance(probs, float):
        probs = [probs, 1.0 - probs]
        
    prob_fake = float(probs[0])
    prob_true = float(probs[1])
    etiqueta = "FALSO" if prob_fake >= prob_true else "VERDADERO"
    return etiqueta, prob_fake, prob_true

def evaluar_sensacionalismo(texto):
    """
    Clasifica si un texto o párrafo es Sensacionalista usando Bert Spanish Sensationalism.
    """
    res = sensacionalismo_classifier(str(texto))[0]
    score_sens = next(item["score"] for item in res if item["label"] in ["LABEL_1", "Sensacionalista"])
    etiqueta = "Sensacionalista" if score_sens >= 0.50 else "NO Sensacional"
    return etiqueta, float(score_sens)

def evaluar_redundancia(texto_general, lista_parrafos, umbral=0.70):
    """
    Calcula la redundancia global de la noticia y la redundancia relativa de cada párrafo con SBERT.
    """
    if len(lista_parrafos) < 2:
        return {
            "global_label": "NO Redundante",
            "global_score": 0.0,
            "parrafos_scores": {i: (0.0, "NO Redundante") for i in range(len(lista_parrafos))}
        }
    
    embeddings = sbert_model.encode(lista_parrafos, convert_to_tensor=True)
    matriz_sim = util.cos_sim(embeddings, embeddings).cpu().numpy()
    
    n = len(lista_parrafos)
    similitudes_maximas_parrafo = []
    detalles_parrafos = {}
    
    for i in range(n):
        sims_otros = [matriz_sim[i][j] for j in range(n) if i != j]
        max_sim = float(np.max(sims_otros)) if sims_otros else 0.0
        lbl_p = "Redundante" if max_sim >= umbral else "NO Redundante"
        detalles_parrafos[i] = (max_sim, lbl_p)
        similitudes_maximas_parrafo.append(max_sim)
        
    score_global = float(np.mean(similitudes_maximas_parrafo)) if similitudes_maximas_parrafo else 0.0
    label_global = "Redundante" if score_global >= umbral else "NO Redundante"
    
    return {
        "global_label": label_global,
        "global_score": score_global,
        "parrafos_scores": detalles_parrafos
    }


# 🚀 5. Función Principal del Pipeline (`ejecutar_pipeline_tesis`)
Esta función integra todos los módulos y genera los registros (logs con emojis) para seguir el proceso completo desde la noticia cruda hasta la clasificación final.


In [ ]:
def ejecutar_pipeline_tesis(noticia_texto, top_k_filtro=2, umbral_score_falsedad=0.50):
    print("=" * 80)
    print(" 📰 [ NOTICIA / TEXTO COMPLETO RECIBIDO ]")
    print("=" * 80)
    print(f"{noticia_texto.strip()}\n")
    
    # 1. MEC GEN
    print("⚙️  Ejecutando [ MEC GEN ] - Extracción de párrafos...")
    parrafos = mec_gen_extraer_parrafos(noticia_texto)
    print(f"👉 Se extrajeron {len(parrafos)} párrafos/oraciones para análisis individual:\n")
    for idx, p in enumerate(parrafos, 1):
        print(f"   ▫️ Párrafo {idx} (P{idx}): \"{p}\"")
    print("\n" + "-" * 80)
    
    # 2. MODELO CLASIFICADOR PRINCIPAL (SaBERT)
    print("🤖 [ MODELO CLASIFICADOR PRINCIPAL - SaBERT ] (Detección de Falsedad)")
    print("-" * 80)
    etiqueta_gen, prob_fake_gen, prob_true_gen = evaluar_sabert_falsedad(noticia_texto)
    print(f"📊 [ ANÁLISIS GENERAL ]")
    print(f"   - Noticia en General: [{etiqueta_gen}] (Prob. Falso: {prob_fake_gen:.4f} | Prob. Verdadero: {prob_true_gen:.4f})")
    
    print(f"\n📋 [ ANÁLISIS POR PÁRRAFO ]")
    evaluaciones_parrafos = []
    for idx, p in enumerate(parrafos, 1):
        lbl_p, p_fake, p_true = evaluar_sabert_falsedad(p)
        evaluaciones_parrafos.append({
            'id': idx,
            'nombre': f'P{idx}',
            'texto': p,
            'etiqueta': lbl_p,
            'score_fake': p_fake,
            'score_true': p_true
        })
        print(f"   - P{idx}: {lbl_p} (Score Falsedad: {p_fake:.4f})")
    print("\n" + "-" * 80)
    
    # 3. FILTRO DE PÁRRAFOS SCORE MÁS ALTO
    print("🎯 (Filtro: Pasan los párrafos con el SCORE MÁS ALTO de falsedad)")
    parrafos_ordenados = sorted(evaluaciones_parrafos, key=lambda x: x['score_fake'], reverse=True)
    parrafos_filtrados = [p for p in parrafos_ordenados if p['score_fake'] >= umbral_score_falsedad][:top_k_filtro]
    
    if not parrafos_filtrados and parrafos_ordenados:
        parrafos_filtrados = parrafos_ordenados[:top_k_filtro]
        
    parrafos_filtrados = sorted(parrafos_filtrados, key=lambda x: x['id'])
    
    print(f"👉 Párrafos Seleccionados ({len(parrafos_filtrados)}):")
    for pf in parrafos_filtrados:
        print(f"   ✅ {pf['nombre']} (Score Falsedad: {pf['score_fake']:.4f}): \"{pf['texto']}\"")
    print("\n" + "=" * 80)
    
    # 4. MODELOS SENSACIONALISMO Y REDUNDANCIA
    print(" [ PÁRRAFOS FILTRADOS + NOTICIA COMPLETA ENTRAN A MODELOS SECUNDARIOS ]")
    print("=" * 80 + "\n")
    
    print("=============================")
    print("[   MODELO CLASIFICADOR     ]")
    print("[     SENSACIONALISMO       ]")
    print("=============================")
    lbl_sens_gen, score_sens_gen = evaluar_sensacionalismo(noticia_texto)
    print(f"+-> Texto General: [{lbl_sens_gen}] (Score: {score_sens_gen:.4f})")
    for pf in parrafos_filtrados:
        lbl_sens_p, score_sens_p = evaluar_sensacionalismo(pf['texto'])
        print(f"+-> {pf['nombre']} ({pf['nombre']}): [{lbl_sens_p}] (Score: {score_sens_p:.4f})")
        
    print("\n" + "-" * 60 + "\n")
    
    print("=============================")
    print("[   MODELO CLASIFICADOR     ]")
    print("[       REDUNDANCIA         ]")
    print("=============================")
    res_redundancia = evaluar_redundancia(noticia_texto, parrafos)
    print(f"+-> Texto General: [{res_redundancia['global_label']}] (Score Similitud: {res_redundancia['global_score']:.4f})")
    for pf in parrafos_filtrados:
        idx_p = pf['id'] - 1
        score_red_p, lbl_red_p = res_redundancia['parrafos_scores'].get(idx_p, (0.0, "NO Redundante"))
        print(f"+-> {pf['nombre']} ({pf['nombre']}): [{lbl_red_p}] (Score Redundancia: {score_red_p:.4f})")
        
    print("\n" + "=" * 80)
    print("✨ PIPELINE FINALIZADO EXITOSAMENTE ✨")
    print("=" * 80)


# 🧪 6. Ejecución del Pipeline con Noticia de Ejemplo


In [ ]:
noticia_ejemplo = """
¡Escándalo total en la economía! Un informe confidencial filtrado por hackers revela que el Banco Central oculta la verdad y provocará la quiebra nacional mañana mismo.
Los analistas de la institución financiera señalan que las medidas tomadas buscan estabilizar los niveles de inflación gradualmente durante el próximo trimestre.
Expertos advierten que el colapso financiero es inminente y que el apocalipsis económico destruirá los ahorros de toda la población en cuestión de horas.
"""

ejecutar_pipeline_tesis(noticia_ejemplo)


# 🧭 7. Módulo de Explicabilidad Forense: Las 5 Dimensiones del IML y Arquetipos Discursivos Flexibles
A partir de los modelos de Nivel 1 (SaBERT, BETO Sensacionalismo y SBERT Redundancia), se calcula el **Índice de Manipulación Lingüística (IML)** en 5 dimensiones normalizadas de [0 a 100]:
- **D1 (Carga Emocional):** Grado de dramatismo oracional general asignado por BETO.
- **D2 (Volatilidad de Gatillo):** Desbalance causal entre la frase detonante y la media del artículo.
- **D3 (Amortiguación Contextual):** Eficacia del cuerpo textual en diluir la alarma inicial.
- **D4 (Reiteración y Redundancia):** Bucle circular de premisas mediante similitud semántica SBERT.
- **D5 (Cohesión y Fluidez):** Integridad sintáctica y progresión proposicional en la Banda de Shannon.

La clasificación en los **5 Arquetipos Discursivos** evalúa la afinidad morfoestilométrica continua (rampas difusas [0, 1]), evitando sesgos de umbrales rígidos y reportando el arquetipo dominante, el secundario y el desglose de afinidades.


In [ ]:
def ramp_up(x, lo, hi):
    if x <= lo: return 0.0
    if x >= hi: return 1.0
    return (x - lo) / (hi - lo)

def ramp_down(x, hi, lo):
    return 1.0 - ramp_up(x, lo, hi)

def bell_band(x, lo, mid_lo, mid_hi, hi):
    if x <= lo or x >= hi: return 0.0
    if mid_lo <= x <= mid_hi: return 1.0
    if x < mid_lo: return (x - lo) / (mid_lo - lo)
    return (hi - x) / (hi - mid_hi)

ARQUETIPOS_META = [
    ("ESTILO_I_DESINFO_ESTRIDENTE", "Estilo I: Desinformación Estridente / Cliché Hiperbólico",
     "Alarma emocional sostenida en todo el texto, sin amortiguación del cuerpo; la repetición circular de premisas agrava la estridencia.",
     "Afinidad = 0.45·(D1 alta) + 0.30·(D3 baja/sin amortiguar) + 0.25·(D4 bucle circular)"),
    ("ESTILO_II_CLICKBAIT_CABECERA", "Estilo II: Cebo Comercial / Clickbait de Cabecera",
     "Titular o apertura alarmista diseñado para capturar clics, pero con cuerpo informativo formal que amortigua y diluye la alarma.",
     "Afinidad = 0.45·(D2 gatillo alto) + 0.40·(D3 amortiguación alta) + 0.15·(D1 moderada/baja)"),
    ("ESTILO_III_PROPAGANDA_SOBRIA", "Estilo III: Propaganda Institucional / Astroturfing Encubierto",
     "Tono formal, sobrio y frío, pero con bucles de repetición semántica calculados para inducir verdad ilusoria.",
     "Afinidad = 0.45·(D4 bucle alto) + 0.35·(D1 sobria/baja) + 0.20·(D2 gatillo bajo/homogéneo)"),
    ("ESTILO_IV_DESARTICULADO", "Estilo IV: Incoherencia Estructural / Generación Rota",
     "Texto con baja conectividad proposicional, fragmentación sintáctica o párrafos desvinculados sin hilo temático.",
     "Afinidad = 0.50·(D5 cohesión rota) + 0.35·(D4 redundancia dispersa) + 0.15·(D3 baja)"),
    ("ESTILO_V_PERIODISMO_EQUILIBRADO", "Estilo V: Periodismo Profesional Balanceado",
     "Redacción equilibrada con variedad léxica, desarrollo temático progresivo en Banda de Shannon y ausencia de estridencias.",
     "Afinidad = 0.35·(D5 cohesión óptima) + 0.30·(D1 sobria) + 0.20·(D4 progresión lineal) + 0.15·(D2 equilibrada)")
]

def clasificar_arquetipo_flexible(d1, d2, d3, d4, d5):
    a1 = 0.45 * ramp_up(d1, 40.0, 75.0) + 0.30 * ramp_down(d3, 60.0, 10.0) + 0.25 * ramp_up(d4, 35.0, 75.0)
    a2 = 0.45 * ramp_up(d2, 35.0, 65.0) + 0.40 * ramp_up(d3, 40.0, 75.0) + 0.15 * ramp_down(d1, 70.0, 30.0)
    a3 = 0.45 * ramp_up(d4, 45.0, 80.0) + 0.35 * ramp_down(d1, 60.0, 20.0) + 0.20 * ramp_down(d2, 65.0, 25.0)
    a4 = 0.50 * ramp_down(d5, 70.0, 38.0) + 0.35 * ramp_down(d4, 55.0, 25.0) + 0.15 * ramp_down(d3, 60.0, 20.0)
    a5 = 0.35 * ramp_up(d5, 45.0, 75.0) + 0.30 * ramp_down(d1, 55.0, 20.0) + 0.20 * bell_band(d4, 20.0, 30.0, 60.0, 75.0) + 0.15 * ramp_down(d2, 55.0, 25.0)
    afs = [a1, a2, a3, a4, a5]
    orden = sorted(range(5), key=lambda i: (-afs[i], i))
    ganador = orden[0]
    segundo = orden[1]
    return {
        "ganador": ARQUETIPOS_META[ganador],
        "afinidad_ganador": round(afs[ganador] * 100, 1),
        "segundo": ARQUETIPOS_META[segundo],
        "afinidad_segundo": round(afs[segundo] * 100, 1),
        "afinidades_completas": {ARQUETIPOS_META[i][0]: round(afs[i] * 100, 1) for i in range(5)}
    }

def auditar_explicabilidad_noticia(texto):
    parrafos = mec_gen_extraer_parrafos(texto)
    k = len(parrafos)
    scores_sens = [evaluar_sensacionalismo(p)[1] for p in parrafos]
    _, p_full = evaluar_sensacionalismo(texto)
    p_max = max(scores_sens) if scores_sens else p_full
    p_mean = float(np.mean(scores_sens)) if scores_sens else p_full
    
    if k > 2:
        scores_sorted = sorted(scores_sens)
        dilution_ratio = scores_sorted[-2] / (p_max + 1e-6)
    else:
        dilution_ratio = 1.0
        
    if k >= 2:
        embs = sbert_model.encode(parrafos, convert_to_tensor=True)
        sim_matrix = util.cos_sim(embs, embs).cpu().numpy()
        np.fill_diagonal(sim_matrix, 0.0)
        max_intra = float(np.max(sim_matrix))
    else:
        max_intra = 0.35
        
    d1 = min(100.0, max(0.0, p_full * 100.0))
    d2 = min(100.0, max(0.0, max(0.0, p_max - p_mean) * 150.0))
    d3 = min(100.0, max(0.0, (max(0.0, 1.0 - dilution_ratio) if p_max >= 0.50 else 1.0) * 100.0))
    d4 = min(100.0, max(0.0, ((max_intra - 0.34) / (0.90 - 0.34)) * 100.0))
    
    if max_intra <= 0.5924:
        d5 = (max_intra / 0.5924) * 60.0
    elif max_intra <= 0.8077:
        d5 = 90.0 + (1.0 - abs(max_intra - 0.625) / 0.18) * 10.0
    else:
        d5 = max(30.0, 100.0 - ((max_intra - 0.8077) / 0.19) * 70.0)
    d5 = min(100.0, max(0.0, d5))
    
    score_global_iml = 0.35 * d1 + 0.20 * d2 + 0.15 * (100.0 - d3) + 0.20 * d4 + 0.10 * (100.0 - d5)
    res_arch = clasificar_arquetipo_flexible(d1, d2, d3, d4, d5)
    
    print("\n" + "=" * 80)
    print("🧭 [ REPORTE FORENSE DE EXPLICABILIDAD ESTILOMÉTRICA - IML 5D ]")
    print("=" * 80)
    print(f"📊 Índice de Manipulación Lingüística (IML Global): {score_global_iml:.1f} / 100")
    print(f"   - D1 (Carga Emocional / Sensacionalismo) : {d1:.1f} / 100")
    print(f"   - D2 (Volatilidad Gatillo / Cebo)       : {d2:.1f} / 100")
    print(f"   - D3 (Amortiguación Contextual)          : {d3:.1f} / 100")
    print(f"   - D4 (Reiteración y Redundancia SBERT)   : {d4:.1f} / 100")
    print(f"   - D5 (Cohesión y Fluidez en Shannon)     : {d5:.1f} / 100")
    print("-" * 80)
    g_id, g_nom, g_desc, g_reg = res_arch["ganador"]
    s_id, s_nom, s_desc, s_reg = res_arch["segundo"]
    print(f"🏆 Arquetipo Discursivo Asignado: {g_nom}")
    print(f"   - Grado de Afinidad Dominante: {res_arch['afinidad_ganador']}%")
    print(f"   - Diagnóstico: {g_desc}")
    print(f"🥈 Segundo Arquetipo Afín: {s_nom} ({res_arch['afinidad_segundo']}%)")
    print("📈 Espectro Completo de Afinidades:")
    for aid, af_val in res_arch["afinidades_completas"].items():
        barra = "█" * int(af_val // 5) + "░" * (20 - int(af_val // 5))
        print(f"   • {aid:<32}: [{barra}] {af_val:.1f}%")
    print("=" * 80)

auditar_explicabilidad_noticia(noticia_ejemplo)
